In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load ranked baseline (EGFR)
ranked_baseline = pd.read_parquet("../../processed/ENSG00000146648_composite_baseline.parquet")

# Load the cleaned contextual layers
signatures   = pd.read_parquet("../../processed/signatures_clean.parquet")
metabolomics = pd.read_parquet("../../processed/metabolomics_clean.parquet")
mirna        = pd.read_parquet("../../processed/mirna_clean.parquet")

print("Ranked Baseline:", ranked_baseline.shape)
print("Signatures:", signatures.shape)
print("Metabolomics:", metabolomics.shape)
print("miRNA:", mirna.shape)

Ranked Baseline: (1581, 30)
Signatures: (1955, 6)
Metabolomics: (928, 227)
miRNA: (734, 952)


In [2]:
# check how cell line IDs are stored before writing the exclusion logic
print("--- Metabolomics ---")
print("Index name:", metabolomics.index.name)
print("First 5 columns:", metabolomics.columns.tolist()[:5])

print("\n--- miRNA ---")
print("Index name:", mirna.index.name)
print("First 5 columns:", mirna.columns.tolist()[:5])

--- Metabolomics ---
Index name: None
First 5 columns: ['CCLE_ID', 'DepMap_ID', '2-aminoadipate', '3-phosphoglycerate', 'alpha-glycerophosphate']

--- miRNA ---
Index name: miRNA
First 5 columns: ['ACH-000698', 'ACH-000489', 'ACH-000431', 'ACH-000707', 'ACH-000509']


In [11]:
def apply_full_exclusion(ranked_df, tables, criteria=None):
    """
    Filters cell lines against user-supplied exclusion criteria.

    tables   : dict of the source tables, e.g. {"signatures": ..., "metabolomics": ..., "mirna": ...}
    criteria : dict of exclusion rules. Supported keys:
                 "msi_max"     : float, exclude MSIScore above this
                 "cin_max"     : float, exclude CIN above this
                 "metabolite"  : (name, threshold) tuple
                 "mirna"       : (name, threshold) tuple

    Signature thresholds use established biological cutoffs. Metabolite and miRNA
    thresholds have no default: there is no biologically defensible universal
    cutoff for "too high" across different metabolites or miRNAs, so the caller
    must supply an absolute value.
    """
    criteria = criteria or {}
    sig_df = tables["signatures"]

    merged = pd.merge(ranked_df, sig_df, left_on="ACH_ID", right_index=True, how="left")
    exclude_cond = pd.Series(False, index=merged.index)

    # genomic instability, established cutoffs
    if "msi_max" in criteria:
        exclude_cond |= (merged["MSIScore"] > criteria["msi_max"]).fillna(False).astype(bool)
    if "cin_max" in criteria:
        exclude_cond |= (merged["CIN"] > criteria["cin_max"]).fillna(False).astype(bool)

    # metabolite, threshold required
    if "metabolite" in criteria:
        name, threshold = criteria["metabolite"]
        metab_df = tables.get("metabolomics")
        if metab_df is None or name not in metab_df.columns:
            raise ValueError(f"Metabolite '{name}' not found in metabolomics data.")
        if threshold is None:
            raise ValueError(f"No threshold supplied for metabolite '{name}'.")
        high = metab_df.set_index("DepMap_ID")[name] > threshold
        exclude_cond |= merged["ACH_ID"].map(high).eq(True)

    # miRNA, threshold required
    if "mirna" in criteria:
        name, threshold = criteria["mirna"]
        mirna_df = tables.get("mirna")
        if mirna_df is None or name not in mirna_df.index:
            raise ValueError(f"miRNA '{name}' not found in miRNA data.")
        if threshold is None:
            raise ValueError(f"No threshold supplied for miRNA '{name}'.")
        high = mirna_df.loc[name] > threshold
        exclude_cond |= merged["ACH_ID"].map(high).eq(True)

    viable = merged[~exclude_cond].copy()
    print(f"Original: {len(merged)} | Excluded: {exclude_cond.sum()} | Remaining: {len(viable)}")
    return viable


tables = {"signatures": signatures, "metabolomics": metabolomics, "mirna": mirna}

viable_candidates = apply_full_exclusion(ranked_baseline, tables,
                                          criteria={"msi_max": 3.0, "cin_max": 0.5})

cols_to_show = ["ACH_ID", "cell_line_name", "evidence_score", "MSIScore", "CIN", "confidence_score"]
display(viable_candidates[cols_to_show].head(10))

Original: 1581 | Excluded: 978 | Remaining: 603


,ACH_ID,cell_line_name,evidence_score,MSIScore,CIN,confidence_score
8,ACH-000528,ABC1,92.656002,1.84,NaN,0.938606
11,ACH-000606,PECAPJ34CLONEC12,94.313810,2.97,0.494775,0.895543
14,ACH-000222,ASPC1,91.668152,2.61,NaN,0.916126
16,ACH-000832,CAL27,94.043726,1.28,NaN,0.888914
17,ACH-000247,OCUM1,85.851502,1.91,0.408069,0.926013
21,ACH-000511,CALU1,86.978347,0.83,0.435122,0.898058
24,ACH-000768,MDAMB231,88.080164,2.14,0.447449,0.865674
27,ACH-000040,U118MG,82.243814,2.56,NaN,0.890342
29,ACH-000367,NCIH226,84.949704,1.20,NaN,0.853426
46,ACH-000260,SKNAS,75.181627,1.73,NaN,0.914666


In [12]:
def recommend_true_biological_twins(target_ach, viable_df, sig_df, metab_df, mirna_df, top_n=10):
    """
    Dynamically checks available omics data, builds a custom feature space, 
    and calculates similarity to find the closest viable alternative.
    """
    # Prep Signatures by moving ModelID to columns
    sig_df_clean = sig_df.reset_index()
    
    # Prep miRNA by transposing, assigning the index name, and resetting
    mirna_t = mirna_df.T
    mirna_t.index.name = "ACH_ID"
    mirna_t = mirna_t.reset_index()
    
    # Check omics data availability for the target cell line
    has_sig = target_ach in sig_df_clean["ModelID"].values
    has_metab = target_ach in metab_df["DepMap_ID"].values
    has_mirna = target_ach in mirna_t["ACH_ID"].values
    
    if not has_sig:
        return f"Cannot compute: Target {target_ach} lacks baseline signature data."
        
    print(f"--- Diagnosing Data Availability for {target_ach} ---")
    
    # Initialize the master dataframe with Signatures as the base
    master_df = sig_df_clean.copy()
    master_df = master_df.rename(columns={"ModelID": "ACH_ID"})
    active_features = [c for c in master_df.columns if c not in ['ACH_ID', 'IsDefaultEntryForModel']]
    
    # Merge additional layers if data exists for the target
    if has_metab:
        print("Target found in Metabolomics. Adding features...")
        master_df = pd.merge(master_df, metab_df, left_on="ACH_ID", right_on="DepMap_ID", how="inner")
        metab_features = [c for c in metab_df.columns if c not in ['CCLE_ID', 'DepMap_ID', 'ACH_ID']]
        active_features += metab_features
    else:
        print("Target missing Metabolomics data. Excluding from calculation.")
        
    if has_mirna:
        print("Target found in miRNA. Adding features...")
        master_df = pd.merge(master_df, mirna_t, on="ACH_ID", how="inner")
        mirna_features = [c for c in mirna_t.columns if c not in ['ACH_ID']]
        active_features += mirna_features
    else:
        print("Target missing miRNA data. Excluding from calculation.")
        
    print(f"\nFinal calculation running across {len(active_features)} valid biological dimensions...")

    # Isolate the profile and mathematical vector for the target cell line
    target_profile = master_df[master_df["ACH_ID"] == target_ach]
    target_vector = target_profile[active_features].fillna(0) 
    
    # Filter viable candidates to those surviving the inner joins
    viable_master = pd.merge(viable_df[["ACH_ID", "cell_line_name", "evidence_score"]], master_df, on="ACH_ID", how="inner")
    
    # Remove the target cell line from the viable pool
    viable_master = viable_master[viable_master["ACH_ID"] != target_ach].copy() 
    viable_vectors = viable_master[active_features].fillna(0)
    
    if viable_vectors.empty:
         return "No viable cell lines remaining with matching omics data."

    # Calculate cosine similarity
    similarities = cosine_similarity(target_vector, viable_vectors)
    
    # Append scores and isolate top candidates
    viable_master["similarity_score"] = similarities[0]
    top_twins = viable_master.sort_values(by="similarity_score", ascending=False).head(top_n)
    
    return top_twins[["ACH_ID", "cell_line_name", "evidence_score", "similarity_score"]]


# Execute the recommendation engine
target_cell_line = "ACH-000431" 
twins = recommend_true_biological_twins(target_cell_line, viable_candidates, signatures, metabolomics, mirna)

print(f"\nTop dynamic alternative recommendations for {target_cell_line}:")
display(twins)

--- Diagnosing Data Availability for ACH-000431 ---
Target found in Metabolomics. Adding features...
Target found in miRNA. Adding features...

Final calculation running across 965 valid biological dimensions...

Top dynamic alternative recommendations for ACH-000431:


,ACH_ID,cell_line_name,evidence_score,similarity_score
171,ACH-000290,NCIH209,30.445385,0.938146
169,ACH-000594,DMS153,44.178239,0.934720
92,ACH-001321,TT,61.038586,0.924273
262,ACH-000382,CORL24,16.166329,0.917200
210,ACH-000743,CORL95,26.538462,0.867935
106,ACH-000790,SHP77,39.922587,0.846900
158,ACH-000830,NCIH1436,47.258543,0.767966
172,ACH-000179,NCIH1618,41.604254,0.715415
214,ACH-000780,NCIH1105,26.365788,0.685703
199,ACH-000136,CHP126,29.783638,0.681441


In [13]:
# check metabolite range before choosing a threshold
metab_col = "lactate" if "lactate" in metabolomics.columns else metabolomics.columns[2]
print(metab_col)
print(metabolomics[metab_col].describe())

# same for a miRNA
mirna_row = "hsa-miR-21" if "hsa-miR-21" in mirna.index else mirna.index[0]
print("\n", mirna_row)
print(mirna.loc[mirna_row].describe())

lactate
count    928.000000
mean       5.820187
std        0.269868
min        4.110226
25%        5.694144
50%        5.859198
75%        5.996382
max        6.631586
Name: lactate, dtype: float64

 hsa-miR-21
count       952.000000
mean       6172.787598
std        9810.112305
min          14.450000
25%        1337.745056
50%        3540.315063
75%        7368.340210
max      188089.265625
Name: hsa-miR-21, dtype: float64


In [15]:
# demonstration thresholds only - chosen from the observed distributions to show
# the branches execute, not because 6.0 lactate or 7400 miR-21 is biologically meaningful

viable_metab = apply_full_exclusion(ranked_baseline, tables,
    criteria={"msi_max": 3.0, "cin_max": 0.5, "metabolite": ("lactate", 6.0)})

viable_mirna = apply_full_exclusion(ranked_baseline, tables,
    criteria={"msi_max": 3.0, "cin_max": 0.5, "mirna": ("hsa-miR-21", 7400)})

viable_both = apply_full_exclusion(ranked_baseline, tables,
    criteria={"msi_max": 3.0, "cin_max": 0.5,
              "metabolite": ("lactate", 6.0), "mirna": ("hsa-miR-21", 7400)})

# confirm the guard rail fires when a threshold is missing
try:
    apply_full_exclusion(ranked_baseline, tables,
        criteria={"metabolite": ("lactate", None)})
except ValueError as e:
    print("\nValueError raised as expected:", e)

Original: 1581 | Excluded: 1040 | Remaining: 541
Original: 1581 | Excluded: 1055 | Remaining: 526
Original: 1581 | Excluded: 1103 | Remaining: 478

ValueError raised as expected: No threshold supplied for metabolite 'lactate'.
